# Bibliotecas necessárias

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib
import numpy as np

URI = r'C:\Users\Usuario\Documents\Estudos\FIAP\Fase4\raw\dados_irrigacao.csv'

# Criar a variável rendimento_esperado e pré-processamento

In [2]:
def criar_yield_simulation(df): 
    """ 
    Cria uma coluna 'rendimento_esperado' baseada em nutrientes (N, P, K) e Umidade.
    Adiciona ruído aleatório para simular variações naturais do campo.
    """  

    df['rendimento_esperado'] = (
       15 * df['nitrogenio'] + 
       10 * df['fosforo'] + 
       8 * df['postassio'] + 
       2 * df['ph'] + 
       (100 - (df['umidade'] - 50)**2 / 50) + 
       np.random.normal(loc=0, scale=5, size=len(df))
    )

    df['rendimento_esperado'] = df['rendimento_esperado'].clip(lower=0)
    print("Coluna 'rendimento_esperado' simulada e adicionada ao DataFrame.")
    return df

def carregar_processar(URI): 
    """Carrega dados, trata NAs e cria features de sazonalidade."""
    df = pd.read_csv(URI)
    df = df.fillna(df.mean(numeric_only=True))


    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['mes'] = df['timestamp'].dt.month
    df['hora'] = df['timestamp'].dt.hour

    print(f" Dados carregados e pré-processados. Linhas: {len(df)}")
    return df
   

# FUNÇÕES DE TREINAMENTO E AVALIAÇÃO

In [3]:
def train_and_evaluate(df, features, target): 
    """Treina o modelo de Regressão e calcula métricas."""
    
    # Garantia de que a variável target não está nas features (Data Leakage)
    if target in features:
        raise ValueError(f"O TARGET ('{target}') não pode estar na lista de FEATURES.")
        
    X = df[features]
    y = df[target]

    # Divisão em treino (70%) e teste (30%)
    X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.3, random_state=10)

    # Modelo (Random Forest Regressor)
    model_rf = RandomForestRegressor(
        n_estimators=100, random_state=10, n_jobs=-1, max_depth=10
    ).fit(X=X_treino, y=y_treino)
    
    y_pred = model_rf.predict(X=X_teste)

    # Avaliação
    mae = mean_absolute_error(y_true=y_teste, y_pred=y_pred)
    r2 = r2_score(y_true=y_teste, y_pred=y_pred)

    print('--- Métricas de Desempenho ---')
    print(f'Alvo: {target.upper()}')
    print(f'MAE (Erro Absoluto Médio): {mae:.4f}')
    print(f'R² (Coeficiente de Determinação): {r2:.4f}')

    return model_rf, X_teste, y_teste, y_pred

def salvar_model(modelo, filename): 
    """Salva o modelo treinado."""
    joblib.dump(modelo, filename)
    print(f'Modelo salvo em: {filename}')

# Fluxo

In [4]:
BASE_FEATURES = [
    'nitrogenio', 'fosforo', 'postassio', 'probabilidade_precipitacao', 
    'chuva_3h', 'bloqueio_meteorológico', 'estado_bomba', 'mes', 'hora'
]

MODEL_CONFIGS = [
    {
        'target': 'umidade',
        'features': ['ph'] + BASE_FEATURES 
    },
    {
        'target': 'ph',
        'features': ['umidade'] + BASE_FEATURES
    },
    {
        'target': 'rendimento_esperado', 
        'features': ['umidade', 'ph'] + BASE_FEATURES 
    }
]

In [5]:
try:
    # Etapa 1: Carregamento e Pré-processamento
    data = carregar_processar(URI)
    
    # Etapa 2: Simulação da Variável Alvo Final
    data = criar_yield_simulation(data) 
    
    print("\n--- INICIANDO TREINAMENTO SEQUENCIAL DE 3 MODELOS ---")
    
    # Etapa 3: Treinamento e Salvamento
    for config in MODEL_CONFIGS:
        target_name = config['target']
        feature_list = config['features']
        model_file = f'modelo_regressao_{target_name}.joblib'
        
        print(f"\nTreinando modelo para: {target_name.upper()}")
        
        # Treinamento e Avaliação
        model, X_teste, y_teste, y_pred = train_and_evaluate(
            data, feature_list, target_name
        )
        
        # Salvamento
        salvar_model(model, model_file)
        
        # Salvamento dos dados de teste e y_true para cálculo de métricas no Streamlit (PARTE 2)
        X_teste.to_csv(f'X_teste_{target_name}.csv', index=False)
        y_teste.to_csv(f'y_teste_{target_name}.csv', index=False)
        
    print("\n\nTodos os 3 modelos treinados com sucesso e salvos!")
    print("Pronto para a PARTE 1: Criação do Dashboard Streamlit.")

except FileNotFoundError:
    print(f"\nERRO: Arquivo '{URI}' não encontrado. Verifique o caminho.")
except Exception as e:
    print(f"\nOcorreu um erro geral durante o processamento: {e}")

 Dados carregados e pré-processados. Linhas: 1365000
Coluna 'rendimento_esperado' simulada e adicionada ao DataFrame.

--- INICIANDO TREINAMENTO SEQUENCIAL DE 3 MODELOS ---

Treinando modelo para: UMIDADE
--- Métricas de Desempenho ---
Alvo: UMIDADE
MAE (Erro Absoluto Médio): 7.6031
R² (Coeficiente de Determinação): 0.0776
Modelo salvo em: modelo_regressao_umidade.joblib

Treinando modelo para: PH
--- Métricas de Desempenho ---
Alvo: PH
MAE (Erro Absoluto Médio): 0.5571
R² (Coeficiente de Determinação): 0.0009
Modelo salvo em: modelo_regressao_ph.joblib

Treinando modelo para: RENDIMENTO_ESPERADO
--- Métricas de Desempenho ---
Alvo: RENDIMENTO_ESPERADO
MAE (Erro Absoluto Médio): 3.9932
R² (Coeficiente de Determinação): 0.7497
Modelo salvo em: modelo_regressao_rendimento_esperado.joblib


Todos os 3 modelos treinados com sucesso e salvos!
Pronto para a PARTE 1: Criação do Dashboard Streamlit.
